In [ ]:
! pip install --upgrade xarray zarr gcsfs cftime nc-time-axis

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs

xr.set_options(display_style='html')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
plt.rcParams['figure.figsize'] = 12, 6

In [ ]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
df.head()

In [ ]:
# Sea surface temperature only
df_tos = df[
    (df["variable_id"] == "tos") &
    (df["table_id"] == "Omon")
]

# Keep historical + future scenarios
df_tos = df_tos[
    df_tos["experiment_id"].isin(["historical", "ssp245", "ssp585"])
]

df_tos[["source_id", "experiment_id", "member_id", "grid_label", "zstore"]].head()

In [ ]:
models_by_exp = (
    df_tos
    .groupby("source_id")["experiment_id"]
    .unique()
    .reset_index()
)

models_by_exp

In [ ]:
needed = {"historical", "ssp245", "ssp585"}

valid_models = (
    df_tos.groupby("source_id")["experiment_id"]
    .apply(lambda x: needed.issubset(set(x)))
)

valid_models[valid_models].index.tolist()

In [ ]:
model = valid_models[valid_models].index[0]
model

In [ ]:
df_model = df_tos[df_tos["source_id"] == model]

df_model[["source_id", "experiment_id", "member_id", "grid_label", "zstore"]]

In [ ]:
member = df_model["member_id"].iloc[0]

df_model = df_model[df_model["member_id"] == member]
df_model[["source_id", "experiment_id", "member_id", "zstore"]]

In [ ]:
import xarray as xr
import gcsfs

row = df_model[df_model["experiment_id"] == "historical"].iloc[0]
zstore = row["zstore"]

ds = xr.open_zarr(gcsfs.GCSMap(zstore), consolidated=True)
ds

In [ ]:
ds["tos"]

In [ ]:
datasets = {}

for exp in ["historical", "ssp245", "ssp585"]:
    row = df_model[df_model["experiment_id"] == exp].iloc[0]
    zstore = row["zstore"]
    
    ds_exp = xr.open_zarr(gcsfs.GCSMap(zstore), consolidated=True)
    datasets[exp] = ds_exp

datasets

In [ ]:
for exp, ds_exp in datasets.items():
    print(exp, ds_exp.time.values[0], ds_exp.time.values[-1])

In [ ]:
global_means = []

for exp, ds_exp in datasets.items():
    tos = ds_exp["tos"]

    # Step 1: take spatial average first = much smaller
    monthly_global = tos.mean(dim=["j", "i"], skipna=True)

    # Step 2: then yearly average
    yearly_global = monthly_global.groupby("time.year").mean("time")

    temp_df = yearly_global.to_dataframe(name="mean_tos").reset_index()
    temp_df["experiment_id"] = exp

    global_means.append(temp_df)

df_global = pd.concat(global_means, ignore_index=True)
df_global.head()

In [ ]:
plt.figure(figsize=(10, 6))

for exp in ["historical", "ssp245", "ssp585"]:
    temp = df_global[df_global["experiment_id"] == exp]
    plt.plot(temp["year"], temp["mean_tos"], label=exp)

plt.title("Global Mean Sea Surface Temperature by Scenario")
plt.xlabel("Year")
plt.ylabel("Mean Sea Surface Temperature (°C)")
plt.legend(title="Experiment")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
hist = df_global[df_global["experiment_id"] == "historical"]

plt.figure(figsize=(10, 5))
plt.plot(hist["year"], hist["mean_tos"])

plt.title("Historical Global Mean Sea Surface Temperature")
plt.xlabel("Year")
plt.ylabel("Mean Sea Surface Temperature (°C)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
future = df_global[df_global["experiment_id"].isin(["ssp245", "ssp585"])]

plt.figure(figsize=(10, 6))

for exp in ["ssp245", "ssp585"]:
    temp = future[future["experiment_id"] == exp]
    plt.plot(temp["year"], temp["mean_tos"], label=exp)

plt.title("Projected Sea Surface Temperature Under Future Scenarios")
plt.xlabel("Year")
plt.ylabel("Mean Sea Surface Temperature (°C)")
plt.legend(title="Scenario")
plt.grid(True, alpha=0.3)
plt.show()